# Módulo 05: Statistical Edge

**Quant Trading Accelerator**

---


In [ ]:
# Librerías principales
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Machine learning con PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

## Tabla de Contenidos

1. [Objetivos de Aprendizaje](#learning-objectives)
2. [Fundamentos de Álgebra Matricial](#matrix-algebra-fundamentals)
3. [Multiplicación Matriz-Vector](#matrix-vector-multiplication)
4. [Construyendo un Statistical Edge](#building-a-statistical-edge)
5. [Entrenando un Modelo Lineal](#training-a-linear-model)
6. [Evaluando la Predictibilidad](#evaluating-predictability)
7. [Midiendo el Statistical Edge](#measuring-statistical-edge)
8. [Ejercicios Prácticos](#practical-exercises)
9. [Puntos Clave](#key-takeaways)


In [ ]:
# Crear una matriz 3x3
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

In [ ]:
# Acceder a la fila 0 (primera fila)
matrix[0]

In [ ]:
# Acceder al elemento en fila 0, columna 0
matrix[0][0]

In [ ]:
# Acceder al elemento en fila 1, columna 2
matrix[1][2]

---

## Objetivos de Aprendizaje

Al finalizar este módulo, serás capaz de:

- Realizar operaciones esenciales de álgebra matricial
- Entender la relación: **modelo = statistical edge**, **estrategia = ejecución**
- Entrenar un modelo de regresión lineal usando PyTorch
- Evaluar la predictibilidad del modelo usando precisión direccional
- Medir el statistical edge a través de retornos esperados por operación
- Calcular e interpretar el Sharpe Ratio


In [ ]:
# Usando bucles anidados (forma lenta)
no_rows = len(matrix)
no_cols = len(matrix[0])

matrix_copy = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
for i in range(no_rows):
    for j in range(no_cols):
        matrix_copy[i][j] += 1

matrix_copy

In [ ]:
# Usando NumPy (forma rápida, vectorizada)
A = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
A + 1

In [ ]:
# Las operaciones escalares son conmutativas
1 + A

In [ ]:
# Multiplicación escalar
A * 2

---

## Fundamentos de Álgebra Matricial

Las matrices son arrays 2D de números. Entender las operaciones matriciales es esencial
para machine learning y finanzas cuantitativas.

### ¿Qué es una Matriz?

Una matriz es un arreglo rectangular de números organizados en filas y columnas:

$$A = \begin{bmatrix} a_{11} & a_{12} & a_{13} \\ a_{21} & a_{22} & a_{23} \\ a_{31} & a_{32} & a_{33} \end{bmatrix}$$


In [ ]:
# Matriz de features: 3 muestras, 2 features cada una
X = np.array([
    [-0.1, -0.2],   # Sample 1: [lag_1, lag_2]
    [-0.2, -0.4],   # Sample 2
    [-0.4, -0.8]    # Sample 3
])
X

In [ ]:
# Vector de pesos: un peso por feature
w = np.array([-0.5, -0.1])
w

In [ ]:
# Multiplicación matriz-vector: X @ w
y_hat = np.dot(X, w)
y_hat

### Operaciones Matriz-Escalar

Cuando sumamos un escalar a una matriz, se suma a cada elemento:


In [ ]:
# Verificación manual
w1, w2 = w[0], w[1]
np.array([
    -0.1 * w1 + -0.2 * w2,  # Sample 1
    -0.2 * w1 + -0.4 * w2,  # Sample 2
    -0.4 * w1 + -0.8 * w2   # Sample 3
])

---

## Multiplicación Matriz-Vector

¡Esta es la operación central en machine learning! Mapea features a predicciones.

### El Modelo Lineal

$$\hat{y} = X \cdot w + b$$

Donde:
- $X$ = Matriz de features (n muestras x m features)
- $w$ = Vector de pesos (m features)
- $b$ = Escalar de sesgo
- $\hat{y}$ = Predicciones (n muestras)


In [ ]:
bias = 0.0001
y_hat_with_bias = np.dot(X, w) + bias
y_hat_with_bias

### Entendiendo el Cálculo

Para cada muestra, calculamos el producto punto de features con pesos:

$$\hat{y}_1 = x_{11} \cdot w_1 + x_{12} \cdot w_2 = (-0.1)(-0.5) + (-0.2)(-0.1) = 0.07$$


In [ ]:
# Broadcasting (elemento a elemento) - NO es lo que queremos para modelos lineales
X * w

In [ ]:
# Multiplicación matricial - esto es lo que queremos
np.dot(X, w)

### Agregando Sesgo


In [ ]:
url = 'https://drive.google.com/uc?export=download&id=1qnX9GpiL5Ii1FEnHTIAzWnxNejWnilKp'
btcusdt = pd.read_csv(url, parse_dates=["open_time"], index_col='open_time')

print(f"Data shape: {btcusdt.shape}")
btcusdt.head()

### Broadcasting vs Multiplicación Matricial

**Distinción importante**:
- `X * w` = Multiplicación elemento a elemento (broadcasting)
- `X @ w` o `np.dot(X, w)` = Multiplicación matricial


In [ ]:
# Calcular log returns
btcusdt['close_log_return'] = np.log(btcusdt['close'] / btcusdt['close'].shift())

# Crear features con lag
btcusdt['close_log_return_lag_1'] = btcusdt['close_log_return'].shift(1)
btcusdt['close_log_return_lag_2'] = btcusdt['close_log_return'].shift(2)
btcusdt['close_log_return_lag_3'] = btcusdt['close_log_return'].shift(3)

# Eliminar filas NaN
btcusdt = btcusdt.dropna()
btcusdt[['close_log_return', 'close_log_return_lag_1', 'close_log_return_lag_2', 'close_log_return_lag_3']].head()

Broadcasting (elemento a elemento) - NO es lo que queremos para modelos lineales


In [ ]:
btcusdt[['close_log_return', 'close_log_return_lag_1',
         'close_log_return_lag_2', 'close_log_return_lag_3']].corr()

Multiplicación matricial - esto es lo que queremos


In [ ]:
sns.pairplot(btcusdt[['close_log_return', 'close_log_return_lag_1',
                      'close_log_return_lag_2', 'close_log_return_lag_3']],
             diag_kind='kde')
plt.suptitle('Feature Relationships', y=1.02)
plt.show()

---

## Construyendo un Statistical Edge

### La Idea Clave

- **Modelo** = Statistical edge (capacidad de predecir)
- **Estrategia** = Ejecución del statistical edge

Un buen modelo encuentra patrones; una buena estrategia los explota de forma rentable.

### Statistical Edge = Buen Pronóstico

Si podemos predecir la dirección del movimiento de precio mejor que el azar,
tenemos un statistical edge.

### Cargar Datos OHLC


In [ ]:
# Matriz de features X
X = btcusdt[['close_log_return_lag_1', 'close_log_return_lag_2',
             'close_log_return_lag_3']].values
print(f"X shape: {X.shape}")

In [ ]:
# Vector target y
y = btcusdt['close_log_return'].values
print(f"y shape: {y.shape}")

### Ingeniería de Features: Log Returns y Lags


In [ ]:
def time_split(x, train_size=0.75):
    """Split data chronologically for time series."""
    i = int(len(x) * train_size)
    return x[:i].copy(), x[i:].copy()

btcusdt_train, btcusdt_test = time_split(btcusdt, train_size=0.7)

print(f"Train: {len(btcusdt_train)} samples ({btcusdt_train.index.min()} to {btcusdt_train.index.max()})")
print(f"Test: {len(btcusdt_test)} samples ({btcusdt_test.index.min()} to {btcusdt_test.index.max()})")

### Verificar Correlación Serial


In [ ]:
import random
import os

### Visualizar Relaciones entre Features


### Preparar Features y Target


---

## Entrenando un Modelo Lineal

### Split Train/Test para Series Temporales

**Crítico**: Para series temporales, ¡debemos dividir cronológicamente para evitar look-ahead bias!

```
Tiempo: t0 ---- t1 ---- t2 ---- t3 ---- t4 ---- t5 ---- t6 ---- t7
Train:  [===============================]
Test:                                   [=========================]
```


In [ ]:
# Guardar modelo
torch.save(model.state_dict(), "model.pth")

### Entrenamiento del Modelo con PyTorch


In [ ]:
# Obtener predicciones en el set de test
y_hat = model(X_test)
y_hat_np = y_hat.detach().squeeze().numpy()

btcusdt_test['y_hat'] = y_hat_np
btcusdt_test[['close_log_return', 'y_hat']].head(10)

---
CONFIGURACIÓN DE REPRODUCIBILIDAD
---


In [ ]:
btcusdt_test['dir_signal'] = np.sign(btcusdt_test['y_hat'])
btcusdt_test[['close_log_return', 'y_hat', 'dir_signal']].head(10)

---
PREPARAR DATOS
---


In [ ]:
btcusdt_test['is_won'] = btcusdt_test['dir_signal'] == np.sign(btcusdt_test[target])
da = btcusdt_test['is_won'].mean()
print(f"Directional Accuracy: {da:.2%}")

---
DEFINIR MODELO
---


In [ ]:
btcusdt_test['trade_log_return'] = btcusdt_test['dir_signal'] * btcusdt_test[target]
btcusdt_test[['dir_signal', 'close_log_return', 'is_won', 'trade_log_return']].head(10)

---
BUCLE DE ENTRENAMIENTO
---


In [ ]:
expected_trade_return = btcusdt_test['trade_log_return'].mean()
print(f"Expected Trade Return: {expected_trade_return:.6f}")

has_statistical_edge = expected_trade_return > 0
print(f"Has Statistical Edge: {has_statistical_edge}")

---

## Evaluando la Predictibilidad

### Generar Predicciones


In [ ]:
total_log_return = btcusdt_test['trade_log_return'].sum()
print(f"Total Log Return: {total_log_return:.4f}")

# Convertir a retorno simple
total_return = np.exp(total_log_return)
print(f"Total Return: {total_return:.2%}")

In [ ]:
# Valor final del portafolio
initial_capital = 100
final_value = np.exp(total_log_return) * initial_capital
print(f"${initial_capital:.2f} -> ${final_value:.2f}")

### Agregar Señal Direccional

Convertir predicciones continuas a señales de trading:
- **+1** = Ir Long (apostar a que el precio sube)
- **-1** = Ir Short (apostar a que el precio baja)


In [ ]:
cum_trade_log_returns = btcusdt_test['trade_log_return'].cumsum()

plt.figure(figsize=(15, 6))
cum_trade_log_returns.plot()
plt.title('Cumulative Log Returns')
plt.ylabel('Cumulative Log Return')
plt.xlabel('Time')
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Curva de equity bruta
gross_equity_curve = np.exp(cum_trade_log_returns) * initial_capital

plt.figure(figsize=(15, 6))
gross_equity_curve.plot()
plt.title(f'Equity Curve (Starting Capital: ${initial_capital})')
plt.ylabel('Portfolio Value ($)')
plt.xlabel('Time')
plt.axhline(y=initial_capital, color='r', linestyle='--', alpha=0.5)
plt.show()

### Precisión Direccional

¿Con qué frecuencia predecimos la dirección correcta?


In [ ]:
# Sharpe Ratio bruto (por período)
sharpe_raw = btcusdt_test['trade_log_return'].mean() / btcusdt_test['trade_log_return'].std()
print(f"Raw Sharpe Ratio: {sharpe_raw:.4f}")

In [ ]:
# Sharpe Ratio anualizado
trading_days_per_year = 365
hours_per_day = 24
periods_per_year = trading_days_per_year * hours_per_day

sharpe_annual = sharpe_raw * np.sqrt(periods_per_year)
print(f"Annualized Sharpe Ratio: {sharpe_annual:.2f}")

**Interpretación**: Si DA > 50%, tenemos algo de poder predictivo sobre el azar.

---

## Midiendo el Statistical Edge

### Retornos por Operación

Cuando acertamos la dirección, ganamos. Cuando nos equivocamos, perdemos.

$$\text{Trade Return} = \text{Signal} \times \text{Actual Return}$$


In [ ]:
X = [
    [-0.1, -0.01],
    [0.2, 0.5]
]
w = [-0.5, -0.25]
y_hat = []

# TODO: Write a loop to compute y_hat = X @ w

In [ ]:
# Verificar
expected = [-0.1 * -0.5 + -0.01 * -0.25, 0.2 * -0.5 + 0.5 * -0.25]
print(f"Expected: {expected}")
# print(f"Your result: {y_hat}")

### Statistical Edge = Valor Esperado Positivo


In [ ]:
X = [
    [-0.1, -0.01, -0.2],
    [0.2, 0.5, 0.1]
]
X_transpose = []

# TODO: Transpose X from shape (2,3) to (3,2)

In [ ]:
# Verificar
expected = [[-0.1, 0.2], [-0.01, 0.5], [-0.2, 0.1]]
# X_transpose == expected

**Punto Clave**: ¡Si E[trade return] > 0, tenemos un statistical edge!

### Retorno Total


In [ ]:
y_true = [[0.01, -0.02], [-0.01, -0.03]]
y_hat = [[0.02, -0.03], [0.01, -0.01]]
error = []

# TODO: Calculate error = y_true - y_hat (element-wise)

In [ ]:
# Verificar
expected = [[0.01 - 0.02, -0.02 - (-0.03)], [-0.01 - 0.01, -0.03 - (-0.01)]]
# error == expected

### Curva de Equity
